# Imports and Functions

In [14]:
import numpy as np
import pickle
import os
import pandas as pd

In [15]:
def compute_results(concatenated_matrix):
    factors_num = concatenated_matrix.shape[0]
    samples_num = concatenated_matrix.shape[1]
    scenarios_num = concatenated_matrix.shape[2]
    S = np.zeros((factors_num, scenarios_num))  # Matrix to store sensitivity results

    for factor_index in range(factors_num):
        result = concatenated_matrix[factor_index , :]
        # Compute U and UT
        U = np.sum(result[:, 0] * result[:, 1]) / samples_num
        UT = np.sum(result[:, 0] * result[:, 2]) / samples_num
        # Compute the mean outflow (F0)
        F0 = np.mean(result[:, 0])
        # Compute the variance (V)
        V = np.sum((result[:, 0] - F0) ** 2) / samples_num

        S[factor_index, 0] = 1 - (UT - F0 ** 2) / V  # Total effect of factor
        S[factor_index, 1] = (U - F0 ** 2) / V       # Main effect of factor
        S[factor_index, 2] = S[factor_index, 0] - S[factor_index, 1]
    
    return S


# Load-> Concat-> Compute

In [16]:
# DEFINE SA_reference_param and filename
SA_reference_param = 'Peak Discharge'
EVENTS_AND_BIAS_L = [('20120113', 1.56),
    ('20160108', 1.24),
    ('20180101', 1.29),
    ('20191213', 0.79)]

event_idx = 0
filename_date = EVENTS_AND_BIAS_L[event_idx][0]

date = '_'.join([filename_date[:4], filename_date[4:6], filename_date[6:]])
sim_dir = r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/"
sim_dir = os.path.join(sim_dir, date, 'pickles' + '/')
sim_dir

'D:\\Development\\RESEARCH\\Raanana\\SWMM\\from_radar\\Sensitivity_Analysis/2012_01_13\\pickles/'

In [17]:
# Initialize a variable to store concatenated sensitivity matrices
concatenated_matrices = None

# Iterate over each pickle file in the directory
for file_name in os.listdir(sim_dir):
    if file_name.startswith("results_VB_Norain") and file_name.endswith(".pickle"):
        # Extract the number from the filename
        parts = file_name.split("_")
        if len(parts) == 4:  # Check if the filename has expected format
            try:
                number = int(parts[3].split(".")[0])
                # Check if the number is within the desired range
                if 0 <= number <= 99999:
                    # Load sensitivity results from the pickle file
                    with open(os.path.join(sim_dir, file_name), 'rb') as f:
                        sensitivity_results = pickle.load(f)
                        sensitivity_results = np.array(sensitivity_results)  # Convert to NumPy array
                        # Concatenate along axis 1
                        if concatenated_matrices is None:  # If it's the first iteration
                            concatenated_matrices = sensitivity_results
                        else:
                            concatenated_matrices = np.concatenate((concatenated_matrices, sensitivity_results), axis=1)
            except ValueError:
                print(f"Skipping file {file_name} due to invalid number format.")
        else:
            print(f"Skipping file {file_name} due to unexpected filename format.")


In [18]:
concatenated_matrices

array([[[ 5.93114471,  4.76792622, 12.66745949],
        [ 8.15923309, 23.71531677,  3.08232141],
        [28.3711586 , 13.70573807, 34.28099442],
        [ 2.49693108,  9.0258522 ,  0.94163859],
        [ 9.02421761,  2.07860827, 23.13302612]],

       [[ 5.93114471, 10.49526691,  5.93114471],
        [ 8.15923309,  2.72350812, 24.47971725],
        [28.3711586 , 26.58938217, 26.15860748],
        [ 2.49693108,  3.59247613,  2.49693108],
        [ 9.02421761,  6.09557247,  9.02421761]],

       [[ 5.93114471, 10.65819073,  5.84657717],
        [ 8.15923309, 10.17558765,  7.80852413],
        [28.3711586 , 17.81197548, 28.11470413],
        [ 2.49693108,  3.59994054,  2.49376512],
        [ 9.02421761,  5.77238894,  9.19640541]],

       [[ 5.93114471, 12.4857235 ,  4.84145975],
        [ 8.15923309, 10.05707264,  7.37518644],
        [28.3711586 , 31.31288338, 21.91099548],
        [ 2.49693108,  0.94119531,  9.04501724],
        [ 9.02421761, 23.95965576,  2.0369947 ]],

       [[ 5.

In [19]:
S = compute_results(concatenated_matrices)
column_names = ['Total effect', 'Main effect', 'Interactions importance']
index_names = ['IMP', 'Storage', 'n', 'PCT_Zero', 'CN' ]
sensitivity_results_df = pd.DataFrame(S, columns=column_names, index=index_names)

sensitivity_results_df

,Total effect,Main effect,Interactions importance
IMP,-0.703823,0.167901,-0.871724
Storage,-0.171049,0.777592,-0.948641
n,0.022096,0.315515,-0.293419
PCT_Zero,0.590122,1.652988,-1.062866
CN,0.000000,0.287495,-0.287495
